<a href="https://colab.research.google.com/github/moriyamao/flyrank-ml-internship/blob/main/w03_data_contract_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moriyamao/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

*Skill loaded: `writing-data-contracts` (+ `flyrank-data` for schema/gotchas, `querying-big-datasets` for the DuckDB pattern).*

## 0. Connect to the warehouse

*Paste your HF token into Colab Secrets first (key icon 🔑 → add `HF_TOKEN`), then run this cell as-is.*

In [10]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

MONTH = '2026-03'   # mid-panel month -- never the sealed final month (2026-06)
print('Connected. Working month:', MONTH)

Connected. Working month: 2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**My contract, in one line each:**

- `dim_clients`: one row per pseudonymized client (104 clients total).
- `dim_content`: one row per pseudonymized content item (519,606 items), no dates — it's a lookup table for content metadata, not a time series.
- `fact_content_daily_performance`: one row per **(report_date, client_id, content_id)** — a daily performance snapshot, partitioned by month. I'm working the **2026-03** partition only (mid-panel, per the panel-warning rule — never the sealed final month `2026-06`).
- `fact_content_query_90d`: one row per **(client_id, content_id, query_hash)** — a fixed trailing-90-day window, not a daily series. Its window overlaps the last months of the panel, so it can leak into a label defined on recent days.

For this assignment I'm treating `fact_content_daily_performance` (month=2026-03) joined to `dim_content` and `dim_clients` as my working table: **one row = one content item, on one day, for one client, in March 2026.**

In [11]:
# Row counts + date ranges reconcile with the published schema.
counts = con.sql(f"""
    SELECT 'dim_clients' AS t, COUNT(*) n, NULL::date lo, NULL::date hi FROM {TABLES['dim_clients']}
    UNION ALL
    SELECT 'dim_content', COUNT(*), NULL, NULL FROM {TABLES['dim_content']}
    UNION ALL
    SELECT 'fact_daily_march', COUNT(*), MIN(report_date), MAX(report_date) FROM {TABLES['fact_daily']}
""").df()
counts

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,t,n,lo,hi
0,dim_clients,104,NaT,NaT
1,dim_content,519606,NaT,NaT
2,fact_daily_march,9841378,2026-03-01,2026-03-31


In [12]:
# Grain probe: (report_date, client_id, content_id) should be unique in fact_daily.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily']}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print('Duplicate grain rows found:', len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


,report_date,client_hash_id,content_hash_id,c


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** — knowable before the day I'd predict on, safe to use:
- `impressions`, `clicks`, `sessions`, `engagement_rate`, `scroll_rate`, `avg_position` (`gsc_avg_position`), `ai_traffic_pct` — as of the current or prior day, never the outcome day.
- `word_count`, `content_type`, `content_age_days`, `days_since_last_update` from `dim_content`.

**Label / proxy** — the thing I'd predict, never a feature:
- Any forward-looking decline/growth flag built from `report_date`-ordered performance *after* the prediction point (mirrors the starter CSV's `trend_direction`/`trend_pct` trap — those exist downstream of the metric I'd be predicting, so their warehouse equivalents are off-limits as inputs too).

**Context** — for grouping, joining, splitting, reading, never for the model to learn from:
- `client_id`, `content_id`, `report_date`, `query_hash` — IDs and the date key.

**Excluded**, each with a one-line why:
- `provider_used`, `model_used` — not a model feature per the data dictionary (metadata about content generation, not about search performance).
- Anything from `fact_content_daily_performance_sample` — that table IS the sealed final month; using it at all during development means training/validating inside the test window.
- GA4 columns on rows before a client's `ga4_data_start` — zero-filled with `ga4_data_available = FALSE`, so a raw zero there means "not measured," not "no engagement"; excluded until filtered on that flag.
- Any client name, URL, or private query text — never included anywhere in this notebook or the repo.

In [13]:
# Confirm which flag/columns exist so the classification above matches reality.
cols_daily = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 0").df()
cols_daily[['column_name','column_type']]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [14]:
cols_clients = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_clients']} LIMIT 0").df()
cols_clients[['column_name','column_type']]

,column_name,column_type
0,client_hash_id,VARCHAR
1,is_active,BOOLEAN
2,has_gsc_access,BOOLEAN
3,has_ga4_access,BOOLEAN
4,access_profile,VARCHAR
5,client_created_date,DATE
6,client_updated_date,DATE
7,gsc_data_start,DATE
8,ga4_data_start,DATE


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:
# Missingness per column in the March partition — flat check first.
missing_flat = con.sql(f"""
    SELECT
        AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS pct_missing_position,
        AVG(CASE WHEN ga4_data_available IS FALSE THEN 1.0 ELSE 0 END) AS pct_ga4_unavailable
    FROM {TABLES['fact_daily']}
""").df()
missing_flat

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pct_missing_position,pct_ga4_unavailable
0,0.633074,0.651197


In [17]:
# Same check, but grouped -- confirms missingness is patterned (per the flyrank-data gotcha), not random.
missing_by_client = con.sql(f"""
    SELECT client_hash_id,
           AVG(CASE WHEN client_has_ga4 IS FALSE THEN 1.0 ELSE 0 END) AS pct_ga4_unavailable,
           COUNT(*) AS n_rows
    FROM {TABLES['fact_daily']}
    GROUP BY client_hash_id
    ORDER BY pct_ga4_unavailable DESC
    LIMIT 10
""").df()
missing_by_client

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,pct_ga4_unavailable,n_rows
0,client_a1203ffecad62470,1.0,2325
1,client_795153d5b7850ccf,1.0,78709
2,client_599043c0ff13edea,1.0,22537
3,client_8ae2bfb5aa1ffa1e,1.0,11067
4,client_0797ff3a1fc9a6a5,1.0,8060
5,client_08a6a72ff48e62c0,1.0,851275
6,client_770e8e5faa9cddfe,1.0,6200
7,client_62f4a7e64f5e0096,1.0,756660
8,client_2c32078d69f2cbad,1.0,341
9,client_80ee5b7bd5f4eb89,1.0,6076


In [18]:
# Window check: does every client's history start on the same day, or does depth vary?
# (dim_clients.gsc_data_start should show real spread -- confirms the per-client-window warning.)
window_check = con.sql(f"""
    SELECT MIN(gsc_data_start) AS earliest_start,
           MAX(gsc_data_start) AS latest_start,
           COUNT(DISTINCT gsc_data_start) AS distinct_start_dates
    FROM {TABLES['dim_clients']}
""").df()
window_check

,earliest_start,latest_start,distinct_start_dates
0,2025-01-27,2026-06-02,45


In [23]:
# Query table grain + repeat-column guard: per-content context columns repeat per row,
# so they must be read with ANY_VALUE, never SUM'd.
query_grain = con.sql(f"""
    SELECT client_hash_id, content_hash_id, query_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_query_90d']}
    GROUP BY client_hash_id, content_hash_id, query_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print('Duplicate (client,content,query) rows:', len(query_grain))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (client,content,query) rows: 0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data can never say, observed from the checks above:**

- **History depth is unbalanced across clients** — `gsc_data_start` spreads over a real range, not one shared day, so any global calendar window either truncates long-history clients or fabricates history for short-history ones. Per-client windows are required, not optional.
- **GA4 rows are zero-filled, not zero, before `ga4_data_start`** — confirmed above via `ga4_data_available`. A model trained without filtering on that flag would learn "no engagement" from what is really "not measured yet."
- **A third of clients have little or no usable history** (per the skill card) — this data can't support decisions for those clients yet; it can only flag that fact, not paper over it with imputation.
- **`fact_content_query_90d`'s window overlaps the panel's final months** — so any label built from the last 30 days of the panel cannot safely pull unrestricted columns from this table; only `*_prev30`-style, pre-aligned columns are safe.
- **This is decision-support data, never causal proof.** It can say a page's metrics are *observed* to move a certain way in March 2026 — it cannot say a refresh *caused* a ranking change, and it says nothing about how Google's own systems will behave going forward.
- **No client names, URLs, or private query text appear anywhere in this notebook** — only pseudonymized IDs and aggregated numbers, per the self-check below.

In [24]:
# Quick numeric backing for the "a third of clients have little/no usable history" claim.
thin_history = con.sql(f"""
    SELECT
        SUM(CASE WHEN gsc_data_start IS NULL OR gsc_data_start > DATE '2026-01-01' THEN 1 ELSE 0 END) AS thin_or_missing_clients,
        COUNT(*) AS total_clients
    FROM {TABLES['dim_clients']}
""").df()
thin_history

,thin_or_missing_clients,total_clients
0,64.0,104


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this yourself after adding your `HF_TOKEN` to Colab Secrets**
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.